# ⚡ Colab D — PyTorch Lightning 3-Layer Neural Network
## Nonlinear Regression using `LightningModule` + `Trainer`

### Why Lightning?
- Removes boilerplate (train loop, device handling, logging)
- Structured `training_step`, `validation_step`, `configure_optimizers`
- Same model logic, cleaner code organization


In [ ]:
# ── SECTION 1: Install & Imports ─────────────────────────────────────────────
!pip install lightning -q

import torch
import torch.nn as nn
import lightning as L
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
print(f'Lightning version: {L.__version__}')

In [ ]:
# ── SECTION 2: Data ──────────────────────────────────────────────────────────
np.random.seed(42)
N = 1000
X1 = np.random.uniform(-np.pi, np.pi, N)
X2 = np.random.uniform(-np.pi, np.pi, N)
X3 = np.random.uniform(-2, 2, N)
Y  = np.sin(X1)*np.cos(X2) + X3**2 + np.random.randn(N)*0.1
X  = np.column_stack([X1, X2, X3])

scaler_X = StandardScaler()
scaler_Y = StandardScaler()
X_norm = scaler_X.fit_transform(X)
Y_norm = scaler_Y.fit_transform(Y.reshape(-1,1))

X_t = torch.tensor(X_norm, dtype=torch.float32)
Y_t = torch.tensor(Y_norm, dtype=torch.float32)

# Split 80/20
n_train = int(0.8 * N)
train_ds = TensorDataset(X_t[:n_train], Y_t[:n_train])
val_ds   = TensorDataset(X_t[n_train:], Y_t[n_train:])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64)
print('Data loaders ready')

In [ ]:
# ── SECTION 3: LightningModule Definition ────────────────────────────────────
class Swish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

class LightningThreeLayerNet(L.LightningModule):
    """
    3-layer DNN wrapped in LightningModule.
    Architecture: 3 → 16 → 8 → 1
    """
    def __init__(self, lr=0.01):
        super().__init__()
        self.lr = lr
        self.save_hyperparameters()

        # Network layers
        self.net = nn.Sequential(
            nn.Linear(3, 16), Swish(),
            nn.Linear(16, 8), Swish(),
            nn.Linear(8, 1)
        )
        # He init
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        """Lightning calls this for each mini-batch during training."""
        x, y = batch
        loss = nn.functional.mse_loss(self(x), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        """Lightning calls this for validation batches."""
        x, y = batch
        loss = nn.functional.mse_loss(self(x), y)
        self.log('val_loss', loss, prog_bar=True)

    def configure_optimizers(self):
        """Lightning uses this to set up optimizer + scheduler."""
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.7)
        return [optimizer], [scheduler]

model = LightningThreeLayerNet(lr=0.01)
print(model)

In [ ]:
# ── SECTION 4: Train with Lightning Trainer ──────────────────────────────────
from lightning.pytorch.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=100, mode='min')

trainer = L.Trainer(
    max_epochs=500,
    callbacks=[early_stop],
    log_every_n_steps=10,
    enable_checkpointing=False
)

trainer.fit(model, train_loader, val_loader)
print('Training complete!')

In [ ]:
# ── SECTION 5: Results ───────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    y_pred_np = model(X_t).numpy()
y_pred_inv = scaler_Y.inverse_transform(y_pred_np)
y_true_inv = scaler_Y.inverse_transform(Y_norm)

plt.figure(figsize=(8,5))
plt.scatter(y_true_inv[:200], y_pred_inv[:200], alpha=0.5, s=20)
plt.plot([y_true_inv.min(), y_true_inv.max()],
         [y_true_inv.min(), y_true_inv.max()], 'r--', label='Perfect')
plt.xlabel('True y'); plt.ylabel('Predicted y')
plt.title('True vs Predicted — PyTorch Lightning 3-Layer')
plt.legend(); plt.grid(True); plt.show()
print(f'MSE (original scale): {np.mean((y_true_inv - y_pred_inv)**2):.4f}')